# LLM-jp Playground (NII LLMC) を OpenAI 互換 API で試す (v4)

**v4 で確立した設計**

診断で判明した事実 (v3 まで知られていなかった):

1. **LLM-jp-4 系は本当の thinking モデル** (名前通り)。英語で reasoning してから日本語で final を出す
2. **Playground の non-streaming パスは `reasoning_content` を捨てる** (サーバ側集約のバグ・仕様不整合)
3. **Streaming パスは `delta.reasoning_content` で正しく返る**
4. Qwen3.6 / Gemma 4 も thinking ON (v3 で既に判明)

これを踏まえた v4 の対策:

- **`chat_once` を内部 streaming に切替**: non-streaming で消えていた reasoning を確実に拾う
- **`max_tokens` デフォルトを 8000 に増量**: thinking モデル + 創作タスクへの安全マージン
- **LaTeX レンダリング**: `\\(...\\)` → `$...$` 変換 + `IPython.display.Markdown` で美しく表示
- **`smart_chat` の thinking 無効化ロジック**は維持

エンドポイント: `https://llm-jp-playground.apps.llmc.nii.ac.jp/api/v1`


## 1. 準備 + レンダリングヘルパ

In [ ]:
# !pip install --quiet --upgrade openai httpx


In [ ]:
import os, json, time, textwrap, random, re
from dataclasses import dataclass, field
from typing import Optional, Any, List, Dict
from openai import OpenAI, APIError, APIConnectionError, RateLimitError
from IPython.display import display, Markdown, HTML

BASE_URL = "https://llm-jp-playground.apps.llmc.nii.ac.jp/api/v1"
API_KEY = os.environ.get("LLMJP_API_KEY", "dummy")
client = OpenAI(base_url=BASE_URL, api_key=API_KEY, timeout=300.0)

DEFAULT_MAX_TOKENS = 8000   # thinking + final を確保 (v4 で増量)

print("client ready, base_url =", BASE_URL)
print("DEFAULT_MAX_TOKENS =", DEFAULT_MAX_TOKENS)


In [ ]:
# ===== レンダリングヘルパ =====

def latex_normalize(text: Optional[str]) -> str:
    """\\(...\\) → $...$ / \\[...\\] → $$...$$ に変換"""
    if not text:
        return text or ""
    # display math first (greediness 注意で非貪欲マッチ)
    text = re.sub(r'\\\[(.+?)\\\]', r'$$\1$$', text, flags=re.DOTALL)
    text = re.sub(r'\\\((.+?)\\\)', r'$\1$', text, flags=re.DOTALL)
    return text


def render_md(text: Optional[str]):
    """テキストを LaTeX 正規化して Markdown レンダリング"""
    display(Markdown(latex_normalize(text or "_(empty)_")))


def render_result(r: "ChatResult", label: Optional[str] = None,
                  show_reasoning_head: int = 0):
    """ChatResult を見やすく整形して Markdown レンダリング"""
    head = f"#### {label}" if label else f"#### {r.model}"
    cc = len(r.content) if r.content else 0
    rc = len(r.reasoning) if r.reasoning else 0
    ct = r.usage.get("completion_tokens") if r.usage else None
    meta = (f"`finish={r.finish_reason}` · "
            f"`elapsed={r.elapsed_sec}s` · "
            f"`content={cc}c` · `reasoning={rc}c` · "
            f"`completion_tokens={ct}`")
    parts = [head, "", meta, ""]
    if show_reasoning_head and r.reasoning:
        parts.append(f"<details><summary>reasoning (head {show_reasoning_head}c)</summary>\n\n```\n"
                     + r.reasoning[:show_reasoning_head] + "\n```\n</details>\n")
    if r.content:
        parts.append(latex_normalize(r.content))
    elif r.reasoning:
        parts.append("_⚠️ content was empty, showing reasoning instead:_\n\n"
                     + latex_normalize(r.reasoning))
    else:
        parts.append("_(empty output)_")
    parts.append("\n---")
    display(Markdown("\n".join(parts)))


# 簡易テスト
render_md("これは **テスト** です。$E=mc^2$ と \\(a^2+b^2=c^2\\) と \\[\\int_0^1 x\\,dx = \\frac12\\]")


## 2. ヘルパ関数 — **内部 streaming 版** chat_once

非ストリーミング呼び出しでは Playground が `reasoning_content` を落とすため、
すべての呼び出しを内部的に `stream=True` で実行して delta を集約します。
ユーザー視点では同期的に `ChatResult` が返ってくるので使い勝手は変わりません。


In [ ]:
@dataclass
class ChatResult:
    text: str
    content: Optional[str]
    reasoning: Optional[str]
    finish_reason: Optional[str]
    usage: Optional[dict]
    model: str
    elapsed_sec: float
    raw_last_chunk: Any = field(repr=False, default=None)


def _stream_collect(model, messages, **kwargs):
    """内部 streaming で content と reasoning を集約"""
    kwargs.setdefault("max_tokens", DEFAULT_MAX_TOKENS)
    kwargs["stream"] = True
    # usage は streaming で得るには stream_options が必要
    kwargs.setdefault("stream_options", {"include_usage": True})

    t0 = time.time()
    stream = client.chat.completions.create(model=model, messages=messages, **kwargs)

    content_chunks, reasoning_chunks = [], []
    finish_reason = None
    usage = None
    last_chunk = None
    actual_model = model

    for chunk in stream:
        last_chunk = chunk
        if getattr(chunk, "model", None):
            actual_model = chunk.model
        if getattr(chunk, "usage", None):
            usage = chunk.usage.model_dump() if hasattr(chunk.usage, "model_dump") else dict(chunk.usage)
        if not chunk.choices:
            continue
        ch = chunk.choices[0]
        if ch.finish_reason:
            finish_reason = ch.finish_reason
        delta = ch.delta
        # content delta
        c = getattr(delta, "content", None)
        if c:
            content_chunks.append(c)
        # reasoning delta (キー名のバリエーション)
        for f in ("reasoning_content", "reasoning", "thinking"):
            v = getattr(delta, f, None)
            if v:
                reasoning_chunks.append(v)
                break

    dt = time.time() - t0
    content = "".join(content_chunks) if content_chunks else None
    reasoning = "".join(reasoning_chunks) if reasoning_chunks else None
    if content:
        text = content
    elif reasoning:
        text = f"[reasoning_only] {reasoning}"
    else:
        text = "(empty)"
    return ChatResult(
        text=text, content=content, reasoning=reasoning,
        finish_reason=finish_reason, usage=usage,
        model=actual_model, elapsed_sec=round(dt, 2),
        raw_last_chunk=last_chunk,
    )


def chat_once(model: str, user_msg: str, system: Optional[str] = None, **kwargs) -> ChatResult:
    """素のチャット (内部 streaming, reasoning も拾う)"""
    messages = []
    if system:
        messages.append({"role": "system", "content": system})
    messages.append({"role": "user", "content": user_msg})
    return _stream_collect(model, messages, **kwargs)


def chat_once_nothink(model: str, user_msg: str, system: Optional[str] = None, **kwargs) -> ChatResult:
    """thinking 無効化 (/no_think + extra_body) を試みる"""
    sys_msg = (system or "") + "\n\n/no_think"
    kwargs.setdefault("extra_body", {"chat_template_kwargs": {"enable_thinking": False}})
    messages = [{"role": "system", "content": sys_msg},
                {"role": "user", "content": user_msg}]
    try:
        return _stream_collect(model, messages, **kwargs)
    except Exception:
        kwargs.pop("extra_body", None)
        return _stream_collect(model, messages, **kwargs)


## 3. モデル一覧 + 改良プローブ

`"Hello."` プローブだと thinking モデルでも reasoning を発火しないことが判明したので、
**創作 + 数学の混合プロンプト**で実測する。これで LLM-jp-4 も thinks=True と検出できる。


In [ ]:
models = client.models.list()
model_ids = [m.id for m in models.data]
print("=== all models ===")
for mid in model_ids:
    print(" -", mid)


In [ ]:
PROBE_PROMPT = "1+1 を計算し、結果を一言で答えてください。"  # 短いが思考を誘発する

def probe_model(model: str) -> Dict[str, Any]:
    info = {"model": model, "thinks": None, "nothink_works": None,
            "probe_default": None, "probe_nothink": None}
    try:
        r1 = chat_once(model, PROBE_PROMPT, system="日本語で。",
                       max_tokens=2000, temperature=0.0)
        info["probe_default"] = {
            "content_c": len(r1.content) if r1.content else 0,
            "reasoning_c": len(r1.reasoning) if r1.reasoning else 0,
            "finish": r1.finish_reason,
        }
        info["thinks"] = bool(r1.reasoning)
    except Exception as e:
        info["probe_default_error"] = repr(e)
        return info

    if info["thinks"]:
        try:
            r2 = chat_once_nothink(model, PROBE_PROMPT, system="日本語で。",
                                   max_tokens=1000, temperature=0.0)
            info["probe_nothink"] = {
                "content_c": len(r2.content) if r2.content else 0,
                "reasoning_c": len(r2.reasoning) if r2.reasoning else 0,
                "finish": r2.finish_reason,
            }
            info["nothink_works"] = (not r2.reasoning) and bool(r2.content)
        except Exception as e:
            info["probe_nothink_error"] = repr(e)
    return info


print("=== probing each model... (約 30 秒) ===")
MODEL_BEHAVIOR: Dict[str, Dict[str, Any]] = {}
for m in model_ids:
    print(f"  probing {m}...", end=" ", flush=True)
    info = probe_model(m)
    MODEL_BEHAVIOR[m] = info
    if info["thinks"] is None:
        print("ERROR")
    elif info["thinks"]:
        print(f"thinks=True (rsn={info['probe_default']['reasoning_c']}c, "
              f"ctn={info['probe_default']['content_c']}c), "
              f"nothink_works={info['nothink_works']}")
    else:
        print(f"thinks=False (ctn={info['probe_default']['content_c']}c)")

print()
print("=== summary ===")
for m, info in MODEL_BEHAVIOR.items():
    print(f"  {m:40s}  thinks={info['thinks']}  nothink_works={info.get('nothink_works')}")


## 4. `smart_chat` 自動ルーティング


In [ ]:
def smart_chat(model: str,
               user_msg: str,
               system: Optional[str] = None,
               disable_thinking: bool = True,
               **kwargs) -> ChatResult:
    info = MODEL_BEHAVIOR.get(model, {})
    thinks = info.get("thinks", False)
    nothink_works = info.get("nothink_works", False)

    if not thinks or not disable_thinking:
        return chat_once(model, user_msg, system=system, **kwargs)

    if nothink_works:
        return chat_once_nothink(model, user_msg, system=system, **kwargs)

    # thinking を切れない → 思考分の予算を増やす
    kwargs.setdefault("max_tokens", 12000)
    return chat_once(model, user_msg, system=system, **kwargs)


# 各モデルのルーティング決定をプレビュー
print("=== smart_chat routing ===")
for m, info in MODEL_BEHAVIOR.items():
    if not info.get("thinks"):
        route = "chat_once (素)"
    elif info.get("nothink_works"):
        route = "chat_once_nothink"
    else:
        route = "chat_once + max_tokens=12000"
    print(f"  {m}: {route}")


## 5. 代表モデル変数

In [ ]:
def _pick(substr):
    for m in model_ids:
        if substr.lower() in m.lower():
            return m
    return None

MODEL_LLMJP_SMALL = _pick("llm-jp-4-8b")
MODEL_LLMJP_LARGE = _pick("llm-jp-4-32b")
MODEL_QWEN        = _pick("qwen")
MODEL_GEMMA       = _pick("gemma")

ALL_MODELS = [m for m in (MODEL_LLMJP_SMALL, MODEL_LLMJP_LARGE, MODEL_QWEN, MODEL_GEMMA) if m]
NONTHINK_MODELS = [m for m, b in MODEL_BEHAVIOR.items() if not b.get("thinks")]
THINK_MODELS    = [m for m, b in MODEL_BEHAVIOR.items() if b.get("thinks")]

print("ALL_MODELS     :", ALL_MODELS)
print("NONTHINK_MODELS:", NONTHINK_MODELS)
print("THINK_MODELS   :", THINK_MODELS)


## 6. 基本: 1 ターン (LaTeX レンダリング)


In [ ]:
r = smart_chat(
    MODEL_LLMJP_SMALL,
    "Iwasawa 理論における λ-不変量と μ-不変量を、専門家向けに2段落で。LaTeX を使って構いません。",
    system="日本語で。",
    temperature=0.2,
)
render_result(r, label="LLM-jp 8b — λ/μ 不変量解説", show_reasoning_head=200)


## 7. マルチターン会話 (Markdown レンダリング)


In [ ]:
class ChatSession:
    def __init__(self, model, system=None, disable_thinking=True, **default_kwargs):
        self.model = model
        self.disable_thinking = disable_thinking
        self.default_kwargs = default_kwargs
        self.default_kwargs.setdefault("max_tokens", DEFAULT_MAX_TOKENS)
        self.messages = []
        if system:
            self.messages.append({"role": "system", "content": system})

    def send(self, user_msg, **override) -> ChatResult:
        self.messages.append({"role": "user", "content": user_msg})

        info = MODEL_BEHAVIOR.get(self.model, {})
        thinks = info.get("thinks", False)
        nothink_works = info.get("nothink_works", False)
        kwargs = {**self.default_kwargs, **override}

        msgs = list(self.messages)
        if thinks and self.disable_thinking and nothink_works:
            kwargs.setdefault("extra_body", {"chat_template_kwargs": {"enable_thinking": False}})
            if msgs and msgs[0]["role"] == "system":
                msgs[0] = {"role": "system",
                           "content": msgs[0]["content"] + "\n\n/no_think"}
            else:
                msgs.insert(0, {"role": "system", "content": "/no_think"})
        elif thinks and self.disable_thinking and not nothink_works:
            kwargs.setdefault("max_tokens", 12000)

        try:
            r = _stream_collect(self.model, msgs, **kwargs)
        except Exception:
            kwargs.pop("extra_body", None)
            r = _stream_collect(self.model, msgs, **kwargs)

        # 履歴には content を蓄積 (空なら reasoning で代用)
        reply = r.content or r.reasoning or ""
        self.messages.append({"role": "assistant", "content": reply})
        return r


sess = ChatSession(
    MODEL_LLMJP_SMALL,
    system="あなたは日本語数学アシスタントです。LaTeX を適宜使い、簡潔に答えてください。",
    temperature=0.2,
)
for q in [
    "Chebyshev bias とは何か、3行で説明してください。",
    "では円分体 $\\mathbb{Q}(\\zeta_p)$ の Iwasawa 塔における類似現象は？",
    "代表的な参考文献を1つ挙げてください。",
]:
    display(Markdown(f"**Q:** {q}"))
    r = sess.send(q)
    render_result(r, label=f"A ({r.model})")


## 8. ライブストリーミング (reasoning と content を区切って逐次表示)


In [ ]:
def chat_stream_live(model, user_msg, system=None,
                     disable_thinking=True, show_reasoning=True, **kwargs):
    """ライブで delta を print し、最後に最終本文を Markdown 再レンダリング"""
    info = MODEL_BEHAVIOR.get(model, {})
    thinks = info.get("thinks", False)
    nothink_works = info.get("nothink_works", False)

    sys_msg = system or ""
    if thinks and disable_thinking and nothink_works:
        sys_msg += "\n\n/no_think"
        kwargs.setdefault("extra_body", {"chat_template_kwargs": {"enable_thinking": False}})

    msgs = []
    if sys_msg:
        msgs.append({"role": "system", "content": sys_msg})
    msgs.append({"role": "user", "content": user_msg})
    kwargs.setdefault("max_tokens", DEFAULT_MAX_TOKENS)
    kwargs["stream"] = True

    stream = client.chat.completions.create(model=model, messages=msgs, **kwargs)
    content_chunks, reasoning_chunks = [], []
    in_reasoning = False
    for chunk in stream:
        if not chunk.choices: continue
        delta = chunk.choices[0].delta
        for f in ("reasoning_content", "reasoning", "thinking"):
            v = getattr(delta, f, None)
            if v:
                reasoning_chunks.append(v)
                if show_reasoning:
                    if not in_reasoning:
                        print("--- [reasoning] ---")
                        in_reasoning = True
                    print(v, end="", flush=True)
                break
        c = getattr(delta, "content", None)
        if c:
            if in_reasoning:
                print("\n--- [content] ---")
                in_reasoning = False
            content_chunks.append(c)
            print(c, end="", flush=True)
    print()
    content = "".join(content_chunks)
    reasoning = "".join(reasoning_chunks)
    # 最後に整形して再表示
    if content:
        display(Markdown("**[整形表示]**"))
        render_md(content)
    return {"content": content, "reasoning": reasoning}


_ = chat_stream_live(
    MODEL_LLMJP_SMALL,
    "$\\zeta_p$ を 1 の原始 $p$ 乗根とする。$\\mathbb{Q}(\\zeta_p)/\\mathbb{Q}$ の Galois 群を、$p$ 奇素数の場合について簡潔に述べてください。",
    system="日本語で。LaTeX を使って構いません。",
    temperature=0.2,
)


## 9. 4モデル横並び比較 (LaTeX レンダリング版)


In [ ]:
def compare_models(prompt, system=None, models=None, show_reasoning_head=0, **kwargs):
    models = models or ALL_MODELS
    display(Markdown(f"### 🟦 Prompt\n\n{latex_normalize(prompt)}\n\n---"))
    results = {}
    for m in models:
        try:
            r = smart_chat(m, prompt, system=system, **kwargs)
            results[m] = r
            render_result(r, label=m, show_reasoning_head=show_reasoning_head)
        except Exception as e:
            results[m] = {"error": repr(e)}
            display(Markdown(f"#### {m}\n\n**ERROR**: `{e}`\n\n---"))
    return results


prompt = textwrap.dedent("""\
    次の問題に日本語で答えてください。LaTeX を使って構いません。
    
    『素数 $p$ に対し、$p-1$ を割る最大の 2 ベキを $v_2(p-1)$ と書く。
     $p = 37$ のときの値を求めよ。また 100 以下の素数で $v_2(p-1) \\geq 5$ となる
     $p$ をすべて列挙せよ。』
""").strip()

_ = compare_models(
    prompt,
    system="あなたは日本語数学アシスタントです。手順を簡潔に示し、最後に答えを明示してください。",
    temperature=0.2,
)


## 10. パラメータ実験 (俳句, 温度比較)

LLM-jp-4 の thinking を `/no_think` で切って、創作タスクで温度の効きを見る。


In [ ]:
PARAM_MODEL = MODEL_LLMJP_SMALL or NONTHINK_MODELS[0]
print(f"PARAM_MODEL = {PARAM_MODEL}",
      "(thinking, /no_think で抑制)" if PARAM_MODEL in THINK_MODELS else "(non-thinking)")

prompt = "「春の朝、研究室で計算機に向かう」という情景を、五七五の俳句で詠んでください。日本語で本文のみ。"

for T in [0.0, 0.4, 0.8, 1.2]:
    try:
        r = smart_chat(PARAM_MODEL, prompt, temperature=T, max_tokens=2000)
        render_result(r, label=f"temperature = {T}")
    except Exception as e:
        display(Markdown(f"#### temperature = {T}\n\n**ERROR**: `{e}`"))


In [ ]:
# seed (vLLM 等で対応していれば再現性)
for seed in [42, 42, 1234]:
    r = smart_chat(PARAM_MODEL,
                   "1から10までの整数をランダムな順に並べてください。",
                   temperature=0.7, seed=seed, max_tokens=2000)
    display(Markdown(f"**seed={seed}**\n\n{latex_normalize(r.content or '(empty)')}"))


### おまけ: 同じプロンプトを Qwen3.6 で thinking ON/OFF 比較


In [ ]:
if MODEL_QWEN and MODEL_BEHAVIOR.get(MODEL_QWEN, {}).get("thinks"):
    haiku_prompt = "「春の朝、研究室で計算機に向かう」を五七五の俳句で。日本語で本文のみ。"
    r_off = smart_chat(MODEL_QWEN, haiku_prompt, temperature=0.7,
                       max_tokens=2000, disable_thinking=True)
    render_result(r_off, label=f"{MODEL_QWEN} — thinking OFF")
    r_on = smart_chat(MODEL_QWEN, haiku_prompt, temperature=0.7,
                      max_tokens=8000, disable_thinking=False)
    render_result(r_on, label=f"{MODEL_QWEN} — thinking ON",
                  show_reasoning_head=500)


## 11. 研究文脈の小タスク

### 11.1 日本語要約 (4モデル比較)


In [ ]:
abstract_ja = textwrap.dedent("""\
    本研究では、代数体の素イデアル分解における Chebyshev 偏差現象を実験的に解析する。
    特に円分体 $\\mathbb{Q}(\\zeta_p)$ の Iwasawa 塔 $\\{\\mathbb{Q}(\\zeta_{p^{n+1}})\\}$ において、
    相対類数 $h^-$ の $p$-進付値 $v_p(h^-)$ の $n$ に関する増大則と、
    $\\lambda$-不変量との関係を計算機実験で検証した。
    本論文では 15 個の素数、3 つの $\\lambda$-類について、
    $\\sum v_p(B_{1,\\chi}) = \\lambda_p - 1$ という経験則が高精度で成立することを示し、
    これを「経験則 S」と呼ぶ。
    また指数関数的減衰係数 $\\alpha$ が $1/\\sqrt{\\lambda}$ に比例し、
    長距離極限のばらつき $\\delta_\\infty$ がクラス内で普遍的に振る舞うことを観測した。
""").strip()

prompt = f"""次のアブストラクトを、専門家向けに 2 センテンスで要約してください。LaTeX をそのまま使って構いません。

---
{abstract_ja}
---
"""
_ = compare_models(prompt, temperature=0.2)


### 11.2 Generator → Verifier


In [ ]:
GEN_MODEL = MODEL_LLMJP_LARGE or MODEL_LLMJP_SMALL
VER_MODEL = next((m for m in ALL_MODELS if m != GEN_MODEL), None)

display(Markdown(f"**Generator**: `{GEN_MODEL}`  \n**Verifier**: `{VER_MODEL}`"))

gen_prompt = "整数 $n$ に対し、$n^2$ を 8 で割った余りとして取りうる値をすべて挙げ、簡潔に理由を述べてください。"
gen = smart_chat(GEN_MODEL, gen_prompt, temperature=0.0)
render_result(gen, label=f"Generator: {GEN_MODEL}")

if VER_MODEL:
    gen_text = gen.content or gen.reasoning or ""
    ver_prompt = textwrap.dedent(f"""\
        以下は別のモデルが書いた数学的主張です。論理と結論の正しさを、
        「正/誤/部分的に正」のいずれかで評価し、根拠を 3 行以内で述べてください。
        LaTeX を使って構いません。

        ---
        {gen_text}
        ---
    """)
    ver = smart_chat(VER_MODEL, ver_prompt, temperature=0.0)
    render_result(ver, label=f"Verifier: {VER_MODEL}")


## 12. リトライ付き呼び出し

In [ ]:
def chat_with_retry(model, user_msg, *, max_retries=3, backoff_base=2.0, **kwargs):
    for attempt in range(max_retries + 1):
        try:
            return smart_chat(model, user_msg, **kwargs)
        except (RateLimitError, APIConnectionError) as e:
            if attempt == max_retries:
                raise
            wait = backoff_base ** attempt + random.uniform(0, 0.5)
            print(f"[retry {attempt+1}/{max_retries}] {type(e).__name__}: sleep {wait:.1f}s")
            time.sleep(wait)
        except APIError:
            raise

r = chat_with_retry(MODEL_LLMJP_SMALL,
                    "天気予測のシンプルな数式モデルを1つ提示してください。LaTeX を使ってください。",
                    temperature=0.3)
render_result(r, label="リトライ動作確認")


## まとめ

**v4 で確立されたこと**

1. **Playground の non-streaming パスは `reasoning_content` を落とす**バグがある → 内部 streaming で回避
2. **LLM-jp-4 (8b / 32b-a3b) は実は thinking モデル** (英語で思考、日本語で final)
3. プローブは "Hello." では不十分、思考を誘発する数学プロンプトが必要
4. LaTeX レンダリングは `\\(...\\)` → `$...$` 変換 + `IPython.display.Markdown` で十分綺麗に出る

**Playground 運営への報告候補**

- non-streaming で `choices[0].message.content` が null になりつつ `finish_reason='length'`、
  かつ `provider_specific_fields.reasoning` も null になる現象 (LLM-jp-4 で再現)
- 同じプロンプトを streaming で投げると `delta.reasoning_content` が正常に流れてくる
- これは「集約段で reasoning channel をマージし忘れている」典型的なバグの可能性

**研究パイプラインへの示唆**

- LLM-jp-4 の **英語 thinking + 日本語 final** という挙動は学術的にも面白い (多言語推論の研究材料)
- Generator-Verifier に組み込むなら、reasoning フィールドを Verifier に渡すと判定材料が増える
- ただし内部 streaming は若干オーバーヘッドあり (398 chunks/応答 程度)。並列実行時は注意
